# 🚀 GroceryGOD // Automated Market Uplink
This notebook scrapes Shwapno, Chaldal, Meena Bazar, and Othoba, aggregates the data, and pushes it to GitHub Pages.

### 🛠️ Setup Instructions:
1. Go to **Add-ons -> Secrets** in Kaggle.
2. Add a secret with Label `GITHUB_PAT` and your GitHub Personal Access Token as the value.
3. Run all cells.

---
### 📡 Features in this build:
- ✅ Robust step-by-step logging with timestamps
- ✅ Telegram notifications at every major milestone
- ✅ Per-step & total runtime tracking
- ✅ All generated plots auto-sent to Telegram
- ✅ Detailed error reporting with tracebacks to Telegram
- ✅ Final success/failure summary message

In [1]:
# ============================================================
# CELL 0 ▸ INSTALL DEPENDENCIES
# ============================================================
import subprocess, sys, time

_cell_start = time.time()
print('=' * 60)
print('📦  [CELL 0] Installing Dependencies')
print('=' * 60)

def _run(cmd, label):
    print(f'  ⏳  {label}...')
    t = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = time.time() - t
    if r.returncode == 0:
        print(f'  ✅  {label} done  ({elapsed:.1f}s)')
    else:
        print(f'  ❌  {label} FAILED  ({elapsed:.1f}s)')
        print(r.stderr[-800:] if r.stderr else '')
    return r.returncode

_run('pip install -q playwright sqlalchemy aiosqlite requests', 'pip install packages')
_run('playwright install chromium --with-deps', 'playwright install chromium')
_run('apt-get install -y -q sqlite3', 'apt-get install sqlite3')

print(f'\n✅  CELL 0 complete  — {time.time()-_cell_start:.1f}s')
print('=' * 60)

📦  [CELL 0] Installing Dependencies
  ⏳  pip install packages...
  ✅  pip install packages done  (7.4s)
  ⏳  playwright install chromium...
  ✅  playwright install chromium done  (50.2s)
  ⏳  apt-get install sqlite3...
  ✅  apt-get install sqlite3 done  (6.7s)

✅  CELL 0 complete  — 64.3s


In [2]:
# ============================================================
# CELL 1 ▸ TELEGRAM NOTIFIER + GLOBAL LOGGER SETUP
# ============================================================
import os, io, glob, time, logging, traceback, requests
from datetime import datetime, timedelta
from pathlib import Path

# ── Telegram credentials ────────────────────────────────────
TELEGRAM_BOT_TOKEN = '563243860:AAG91WlEnGnplDSSpxYUuZ9YbTytE6ydCJw'
TELEGRAM_CHAT_ID   = '758711653'
TG_API             = f'https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}'

# ── Global run-clock ────────────────────────────────────────
PIPELINE_START     = time.time()
STEP_TIMINGS       = {}   # step_name → elapsed_seconds
STEP_LOG           = []   # list of (emoji, message) for final summary

# ── Python logger ───────────────────────────────────────────
LOG_FORMAT = '%(asctime)s | %(levelname)-8s | %(message)s'
logging.basicConfig(
    level=logging.DEBUG,
    format=LOG_FORMAT,
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('/tmp/grocerygod_run.log', mode='w')
    ]
)
log = logging.getLogger('GroceryGOD')

# ── Telegram helpers ────────────────────────────────────────
def _fmt_dur(seconds: float) -> str:
    """Format seconds → human-readable string."""
    return str(timedelta(seconds=int(seconds)))

def tg_send(text: str, silent: bool = False) -> bool:
    """Send a plain-text message to Telegram. Returns True on success."""
    try:
        r = requests.post(
            f'{TG_API}/sendMessage',
            json={
                'chat_id': TELEGRAM_CHAT_ID,
                'text': text,
                'parse_mode': 'HTML',
                'disable_notification': silent
            },
            timeout=15
        )
        if not r.ok:
            log.warning(f'Telegram sendMessage HTTP {r.status_code}: {r.text[:200]}')
        return r.ok
    except Exception as exc:
        log.warning(f'Telegram sendMessage failed: {exc}')
        return False

def tg_photo(path: str, caption: str = '') -> bool:
    """Send a photo file to Telegram. Returns True on success."""
    try:
        with open(path, 'rb') as fh:
            r = requests.post(
                f'{TG_API}/sendPhoto',
                data={'chat_id': TELEGRAM_CHAT_ID, 'caption': caption[:1024], 'parse_mode': 'HTML'},
                files={'photo': (os.path.basename(path), fh, 'image/png')},
                timeout=30
            )
        if not r.ok:
            log.warning(f'Telegram sendPhoto HTTP {r.status_code}: {r.text[:200]}')
        return r.ok
    except Exception as exc:
        log.warning(f'Telegram sendPhoto failed: {exc}')
        return False

def tg_document(path: str, caption: str = '') -> bool:
    """Send any file as a Telegram document."""
    try:
        with open(path, 'rb') as fh:
            r = requests.post(
                f'{TG_API}/sendDocument',
                data={'chat_id': TELEGRAM_CHAT_ID, 'caption': caption[:1024], 'parse_mode': 'HTML'},
                files={'document': (os.path.basename(path), fh, 'application/octet-stream')},
                timeout=30
            )
        return r.ok
    except Exception as exc:
        log.warning(f'Telegram sendDocument failed: {exc}')
        return False

def tg_send_all_plots(search_dirs=None, caption_prefix='📊 Plot'):
    """Scan directories for PNG/JPG files and send them all to Telegram."""
    if search_dirs is None:
        search_dirs = ['.', '/tmp', os.path.expanduser('~')]
    sent = 0
    patterns = ['**/*.png', '**/*.jpg', '**/*.jpeg', '**/*.svg']
    seen = set()
    for d in search_dirs:
        for pat in patterns:
            for f in glob.glob(os.path.join(d, pat), recursive=True):
                f = os.path.realpath(f)
                if f in seen:
                    continue
                seen.add(f)
                size = os.path.getsize(f)
                if size < 1024:          # skip tiny/empty files
                    continue
                if size > 10 * 1024 * 1024:   # skip files > 10 MB
                    log.warning(f'Skipping oversized plot: {f} ({size//1024}KB)')
                    continue
                cap = f'{caption_prefix}: <code>{os.path.basename(f)}</code>'
                ok = tg_photo(f, cap) if f.lower().endswith(('.png','.jpg','.jpeg')) else tg_document(f, cap)
                if ok:
                    log.info(f'  📤  Sent plot to Telegram: {os.path.basename(f)}')
                    sent += 1
    return sent

# ── Step context manager ────────────────────────────────────
class Step:
    """
    Context manager for a named pipeline step.
    Logs start/end, tracks duration, notifies Telegram, records to STEP_TIMINGS.
    Usage:
        with Step('Scrape Shwapno', emoji='🛒') as s:
            ...do work...
    """
    def __init__(self, name: str, emoji: str = '⚙️', notify: bool = True):
        self.name   = name
        self.emoji  = emoji
        self.notify = notify
        self._t0    = None

    def __enter__(self):
        self._t0 = time.time()
        header = f'{self.emoji}  [{self.name}] — STARTED'
        total_so_far = _fmt_dur(time.time() - PIPELINE_START)
        log.info('─' * 55)
        log.info(header)
        log.info(f'Pipeline running: {total_so_far}')
        if self.notify:
            tg_send(
                f'{self.emoji} <b>{self.name}</b> — started\n'
                f'🕐 Pipeline time: <code>{total_so_far}</code>',
                silent=True
            )
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed = time.time() - self._t0
        total   = time.time() - PIPELINE_START
        STEP_TIMINGS[self.name] = elapsed

        if exc_type is None:
            msg = f'✅  [{self.name}] — OK  ({_fmt_dur(elapsed)})'
            log.info(msg)
            STEP_LOG.append(('✅', self.name, elapsed))
            if self.notify:
                tg_send(
                    f'✅ <b>{self.name}</b> — complete\n'
                    f'⏱ Step: <code>{_fmt_dur(elapsed)}</code>  |  '
                    f'🕐 Total: <code>{_fmt_dur(total)}</code>'
                )
        else:
            tb_str = traceback.format_exc()
            msg = f'❌  [{self.name}] — FAILED  ({_fmt_dur(elapsed)})\n{tb_str}'
            log.error(msg)
            STEP_LOG.append(('❌', self.name, elapsed))
            if self.notify:
                short_tb = tb_str[-1500:]  # Telegram msg limit
                tg_send(
                    f'❌ <b>{self.name}</b> — FAILED\n'
                    f'⏱ Step: <code>{_fmt_dur(elapsed)}</code>  |  '
                    f'🕐 Total: <code>{_fmt_dur(total)}</code>\n\n'
                    f'<pre>{short_tb}</pre>'
                )
            # Re-raise so the notebook cell shows the error
            return False
        return True

def pipeline_summary(success: bool = True):
    """Print + Telegram: full timing summary after pipeline ends."""
    total = time.time() - PIPELINE_START
    status_icon = '🎉' if success else '💀'
    status_word = 'SUCCESS' if success else 'FAILED'

    lines = []
    lines.append(f'{status_icon} <b>GroceryGOD Pipeline — {status_word}</b>')
    lines.append(f'🕐 Total runtime: <code>{_fmt_dur(total)}</code>')
    lines.append('')
    lines.append('<b>Step-by-step breakdown:</b>')
    for (icon, sname, stime) in STEP_LOG:
        lines.append(f'  {icon} {sname}: <code>{_fmt_dur(stime)}</code>')
    lines.append('')
    lines.append(f'📅 Finished: <code>{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}</code>')

    summary_text = '\n'.join(lines)

    # Print to notebook
    log.info('=' * 60)
    log.info(f'PIPELINE {status_word} — total {_fmt_dur(total)}')
    for (icon, sname, stime) in STEP_LOG:
        log.info(f'  {icon}  {sname}: {_fmt_dur(stime)}')
    log.info('=' * 60)

    # Send to Telegram
    tg_send(summary_text)

    # Also send the full log file
    tg_document('/tmp/grocerygod_run.log', '📋 Full run log')


# ── Boot message ────────────────────────────────────────────
log.info('=' * 60)
log.info('🚀  GroceryGOD // Automated Market Uplink — INIT')
log.info(f'📅  Run started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
log.info(f'🤖  Telegram bot verified — chat {TELEGRAM_CHAT_ID}')
log.info('=' * 60)

tg_send(
    f'🚀 <b>GroceryGOD Pipeline — STARTED</b>\n'
    f'📅 <code>{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}</code>\n'
    f'🔧 Kaggle Kernel is live. Scraping Shwapno, Chaldal, Meena Bazar, Othoba.'
)
print('\n✅  Telegram notifier + logger ready.')

2026-04-21 19:27:31,470 | INFO     | ============================================================
2026-04-21 19:27:31,472 | INFO     | 🚀  GroceryGOD // Automated Market Uplink — INIT
2026-04-21 19:27:31,472 | INFO     | 📅  Run started: 2026-04-21 19:27:31
2026-04-21 19:27:31,474 | INFO     | 🤖  Telegram bot verified — chat 758711653
2026-04-21 19:27:31,475 | INFO     | ============================================================
2026-04-21 19:27:31,478 | DEBUG    | Starting new HTTPS connection (1): api.telegram.org:443
2026-04-21 19:27:33,890 | DEBUG    | https://api.telegram.org:443 "POST /bot563243860:AAG91WlEnGnplDSSpxYUuZ9YbTytE6ydCJw/sendMessage HTTP/1.1" 401 58
2026-04-21 19:27:33,891 | WARNING  | Telegram sendMessage HTTP 401: {"ok":false,"error_code":401,"description":"Unauthorized"}

✅  Telegram notifier + logger ready.


In [3]:
# ============================================================
# CELL 2 ▸ CONFIGURATION & GIT SETUP
# ============================================================
import subprocess

def shell(cmd: str, label: str = None, check: bool = True) -> subprocess.CompletedProcess:
    """Run a shell command, log output line-by-line, raise on failure."""
    lbl = label or cmd[:60]
    log.debug(f'  $ {cmd}')
    result = subprocess.run(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True
    )
    for line in result.stdout.splitlines():
        log.debug(f'    {line}')
    if check and result.returncode != 0:
        raise RuntimeError(
            f'Command failed (rc={result.returncode}): {cmd}\n{result.stdout[-800:]}'
        )
    return result


with Step('Configuration & Git Setup', '⚙️'):
    from kaggle_secrets import UserSecretsClient

    log.info('Fetching GITHUB_PAT from Kaggle Secrets...')
    user_secrets = UserSecretsClient()
    PAT = user_secrets.get_secret('GITHUB_PAT')
    REPO_URL = f'https://{PAT}@github.com/ranx-x/GroceryGOD.git'
    log.info('✅  GITHUB_PAT fetched successfully.')

    log.info('Configuring Git identity...')
    shell('git config --global user.email "ran.ragibahnafnehal2@gmail.com"', 'git config email')
    shell('git config --global user.name "ranx-x"', 'git config username')
    log.info('✅  Git identity configured.')

    if not os.path.exists('GroceryGOD'):
        log.info('Repo not found locally — cloning...')
        shell(f'git clone {REPO_URL}', 'git clone', check=True)
        log.info('✅  Repo cloned.')
    else:
        log.info('Repo already exists — pulling latest...')
        shell('git -C GroceryGOD pull origin master', 'git pull', check=False)
        log.info('✅  Repo pulled.')

    os.chdir('GroceryGOD')
    cwd = os.getcwd()
    log.info(f'📂  Working directory: {cwd}')
    log.info(f'📁  Repo contents: {os.listdir(".")}')

2026-04-21 19:27:33,908 | INFO     | ───────────────────────────────────────────────────────
2026-04-21 19:27:33,909 | INFO     | ⚙️  [Configuration & Git Setup] — STARTED
2026-04-21 19:27:33,910 | INFO     | Pipeline running: 0:00:02
2026-04-21 19:27:33,911 | DEBUG    | Starting new HTTPS connection (1): api.telegram.org:443
2026-04-21 19:27:34,561 | DEBUG    | https://api.telegram.org:443 "POST /bot563243860:AAG91WlEnGnplDSSpxYUuZ9YbTytE6ydCJw/sendMessage HTTP/1.1" 401 58
2026-04-21 19:27:34,562 | WARNING  | Telegram sendMessage HTTP 401: {"ok":false,"error_code":401,"description":"Unauthorized"}
2026-04-21 19:27:34,571 | INFO     | Fetching GITHUB_PAT from Kaggle Secrets...
2026-04-21 19:27:34,814 | INFO     | ✅  GITHUB_PAT fetched successfully.
2026-04-21 19:27:34,815 | INFO     | Configuring Git identity...
2026-04-21 19:27:34,817 | DEBUG    |   $ git config --global user.email "ran.ragibahnafnehal2@gmail.com"
2026-04-21 19:27:34,843 | DEBUG    |   $ git config --global user.name 

In [4]:
# ============================================================
# CELL 3 ▸ EXECUTE SCRAPERS
# ============================================================
import subprocess

def run_scraper(label: str, cwd_path: str, script: str = 'scraper.py'):
    """
    Run a scraper script in a sub-directory.
    Streams output to log. Returns (success, output_text, elapsed).
    """
    t0 = time.time()
    full_path = os.path.join(os.getcwd(), cwd_path) if not os.path.isabs(cwd_path) else cwd_path
    log.info(f'   📂  Entering: {full_path}')

    if not os.path.isdir(full_path):
        msg = f'Directory not found: {full_path}'
        log.error(f'   ⚠️  {msg}')
        return False, msg, 0.0

    scraper_file = os.path.join(full_path, script)
    if not os.path.isfile(scraper_file):
        msg = f'Scraper file not found: {scraper_file}'
        log.error(f'   ⚠️  {msg}')
        return False, msg, 0.0

    log.info(f'   ▶️  Running: python {script} in {cwd_path}')
    result = subprocess.run(
        f'python {script}',
        cwd=full_path,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=1200   # 20-min per scraper safety timeout
    )
    elapsed = time.time() - t0

    # Print last 40 lines of scraper output
    out_lines = result.stdout.splitlines()
    tail = out_lines[-40:] if len(out_lines) > 40 else out_lines
    for line in tail:
        log.debug(f'    | {line}')

    success = result.returncode == 0
    status  = '✅' if success else '❌'
    log.info(f'   {status}  {label} finished — {_fmt_dur(elapsed)} — rc={result.returncode}')
    return success, result.stdout, elapsed


SCRAPERS = [
    ('🛒  Shwapno',    'swapnoTRACKER'),
    ('🟢  Chaldal',    'PRICETRACKER'),
    ('🏪  Meena Bazar','MEENAtracker/backend'),
    ('🔵  Othoba',     'othobaTRACKER/backend'),
]

scraper_results = {}

with Step('All Market Scrapers', '🛰️'):
    log.info(f'Running {len(SCRAPERS)} scrapers sequentially...')
    tg_send(f'🛰️ <b>Starting {len(SCRAPERS)} Market Scrapers</b>\n' +
            '\n'.join([f'  • {name}' for name, _ in SCRAPERS]))

    all_ok = True
    for name, path in SCRAPERS:
        log.info('')
        log.info(f'── {name} ──────────────────────────')

        ok, output, dur = run_scraper(name, path)
        scraper_results[name] = {'ok': ok, 'duration': dur}

        if ok:
            tg_send(
                f'✅ <b>{name}</b> scraped\n'
                f'⏱ <code>{_fmt_dur(dur)}</code>  |  '
                f'🕐 Total: <code>{_fmt_dur(time.time()-PIPELINE_START)}</code>',
                silent=True
            )
        else:
            all_ok = False
            tg_send(
                f'⚠️ <b>{name}</b> scraper had issues\n'
                f'Check log for details.\n'
                f'<pre>{output[-800:]}</pre>'
            )

    # Summary table in log
    log.info('')
    log.info('── Scraper Summary ──────────────────────────')
    for name, res in scraper_results.items():
        icon = '✅' if res['ok'] else '❌'
        log.info(f'  {icon}  {name}: {_fmt_dur(res["duration"])}')

    # Send any plots generated during scraping
    log.info('📸  Checking for plots generated by scrapers...')
    n_plots = tg_send_all_plots(search_dirs=['.', '/tmp'], caption_prefix='📊 Scraper Plot')
    if n_plots:
        log.info(f'  📤  {n_plots} plots sent to Telegram.')
    else:
        log.info('  ℹ️  No plots found at this stage.')

    if not all_ok:
        log.warning('One or more scrapers reported issues — continuing to aggregator anyway.')

2026-04-21 19:27:40,697 | INFO     | ───────────────────────────────────────────────────────
2026-04-21 19:27:40,698 | INFO     | 🛰️  [All Market Scrapers] — STARTED
2026-04-21 19:27:40,699 | INFO     | Pipeline running: 0:00:09
2026-04-21 19:27:40,701 | DEBUG    | Starting new HTTPS connection (1): api.telegram.org:443
2026-04-21 19:27:41,316 | DEBUG    | https://api.telegram.org:443 "POST /bot563243860:AAG91WlEnGnplDSSpxYUuZ9YbTytE6ydCJw/sendMessage HTTP/1.1" 401 58
2026-04-21 19:27:41,318 | WARNING  | Telegram sendMessage HTTP 401: {"ok":false,"error_code":401,"description":"Unauthorized"}
2026-04-21 19:27:41,319 | INFO     | Running 4 scrapers sequentially...
2026-04-21 19:27:41,321 | DEBUG    | Starting new HTTPS connection (1): api.telegram.org:443
2026-04-21 19:27:41,937 | DEBUG    | https://api.telegram.org:443 "POST /bot563243860:AAG91WlEnGnplDSSpxYUuZ9YbTytE6ydCJw/sendMessage HTTP/1.1" 401 58
2026-04-21 19:27:41,939 | WARNING  | Telegram sendMessage HTTP 401: {"ok":false,"err

TimeoutExpired: Command 'python scraper.py' timed out after 1200 seconds

In [ ]:
# ============================================================
# CELL 4 ▸ RUN AGGREGATOR
# ============================================================
import glob as _glob

with Step('GODdata Aggregator', '🧬'):
    log.info('Checking aggregator.py exists...')
    if not os.path.isfile('aggregator.py'):
        raise FileNotFoundError('aggregator.py not found in repo root!')
    log.info('✅  aggregator.py found.')

    # Snapshot PNGs before aggregator runs so we can diff after
    _pngs_before = set(_glob.glob('**/*.png', recursive=True))

    log.info('▶️  Executing aggregator.py...')
    _agg_t = time.time()
    _agg_result = subprocess.run(
        'python aggregator.py',
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=900  # 15-min safety timeout
    )
    _agg_elapsed = time.time() - _agg_t

    # Log all aggregator output
    log.info(f'aggregator.py output ({len(_agg_result.stdout.splitlines())} lines):')
    for line in _agg_result.stdout.splitlines():
        log.debug(f'  | {line}')

    if _agg_result.returncode != 0:
        raise RuntimeError(
            f'aggregator.py failed (rc={_agg_result.returncode}):\n{_agg_result.stdout[-1000:]}'
        )

    log.info(f'✅  aggregator.py completed in {_fmt_dur(_agg_elapsed)}')

    # Find NEW plots produced by aggregator and send them
    _pngs_after  = set(_glob.glob('**/*.png', recursive=True))
    _new_plots   = _pngs_after - _pngs_before
    log.info(f'📊  {len(_new_plots)} new plot(s) found after aggregation.')

    if _new_plots:
        tg_send(f'🧬 <b>Aggregator complete</b> — {len(_new_plots)} new chart(s) generated:')
        for _plot in sorted(_new_plots):
            _size = os.path.getsize(_plot)
            log.info(f'  📤  Sending: {_plot} ({_size//1024}KB)')
            tg_photo(
                _plot,
                caption=f'📊 <b>GroceryGOD Chart</b>\n<code>{os.path.basename(_plot)}</code>'
            )
    else:
        # Broad scan for any plots anywhere
        _n = tg_send_all_plots(search_dirs=['.', '/tmp'], caption_prefix='📊 Aggregator Plot')
        if _n:
            log.info(f'  📤  Sent {_n} plot(s) from broad scan.')

In [ ]:
# ============================================================
# CELL 5 ▸ PUSH TO GITHUB
# ============================================================
from datetime import datetime

with Step('GitHub Push', '🚀'):
    _now    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    _commit = f'🤖 Automated Market Update: {_now}'

    log.info('Staging all changed files...')
    _r = subprocess.run('git add .', shell=True, capture_output=True, text=True)
    log.debug(f'git add: rc={_r.returncode}')

    log.info('Checking git status...')
    _status = subprocess.run('git status --short', shell=True, capture_output=True, text=True)
    _changed_files = [l for l in _status.stdout.splitlines() if l.strip()]
    log.info(f'  {len(_changed_files)} file(s) changed.')
    for f in _changed_files[:20]:          # log first 20
        log.debug(f'    {f}')

    if not _changed_files:
        log.info('ℹ️  Nothing to commit — working tree clean.')
        tg_send('ℹ️ <b>GitHub Push</b> — nothing to commit, working tree clean.')
    else:
        log.info(f'Committing: "{_commit}"')
        _r = subprocess.run(
            f'git commit -m "{_commit}"',
            shell=True, capture_output=True, text=True
        )
        log.debug(_r.stdout.strip())
        if _r.returncode != 0:
            raise RuntimeError(f'git commit failed:\n{_r.stderr}')

        log.info('Pushing to origin/master...')
        _r = subprocess.run(
            'git push origin master --force',
            shell=True, capture_output=True, text=True
        )
        log.debug(_r.stdout.strip())
        if _r.returncode != 0:
            raise RuntimeError(f'git push failed:\n{_r.stderr}')

        LIVE_URL = 'https://ranx-x.github.io/GroceryGOD'
        log.info(f'✅  Push successful! Live at: {LIVE_URL}')
        tg_send(
            f'🚀 <b>GitHub Push Successful!</b>\n'
            f'📝 Commit: <code>{_commit}</code>\n'
            f'🌐 Live: {LIVE_URL}\n'
            f'📁 Changed files: <code>{len(_changed_files)}</code>'
        )

In [ ]:
# ============================================================
# CELL 6 ▸ FINAL SUMMARY + CLOSE
# ============================================================

# ── Send any remaining / missed plots ──────────────────────
log.info('📸  Final plot sweep — sending all discovered PNGs...')
_total_plots = tg_send_all_plots(
    search_dirs=['.', '..', '/tmp', os.path.expanduser('~')],
    caption_prefix='📊 Final Plot'
)
log.info(f'  ✅  Total plots sent: {_total_plots}')

# ── Pipeline summary ───────────────────────────────────────
pipeline_summary(success=True)

# ── Print final banner ─────────────────────────────────────
_total_runtime = time.time() - PIPELINE_START
print()
print('╔══════════════════════════════════════════════════════╗')
print('║     🎉  GroceryGOD Pipeline FINISHED SUCCESSFULLY    ║')
print(f'║     ⏱  Total Runtime : {_fmt_dur(_total_runtime):<31}║')
print(f'║     📅  Completed    : {datetime.now().strftime("%Y-%m-%d %H:%M:%S"):<31}║')
print(f'║     📊  Plots Sent   : {str(_total_plots):<31}║')
print(f'║     🌐  Live URL     : https://ranx-x.github.io/GroceryGOD ║')
print('╚══════════════════════════════════════════════════════╝')